# 🔬 EndoAgents — Day 1-2: Gemma 4 Zero-Shot Baseline

**Goal:** Establish the zero-shot baseline for Gemma 4 on adenomyosis ultrasound captioning.  
**Output:** Feature coverage scores + raw captions saved to `evaluation/results/baseline_gemma4.json`

**Colab setup:** Runtime → Change Runtime Type → **T4 GPU**

---
**Model options for free tier:**
| Model | VRAM (4-bit) | Notes |
|---|---|---|
| `google/gemma-4-E2B-it` | ~5 GB | Fastest, audio+image |
| `google/gemma-4-E4B-it` | ~8 GB | **Recommended** — best balance |


## Step 0 — Clone repo, fix Python path & install
⚠️ **Run this cell first, every session.** It clones the repo, adds it to `sys.path`, and installs all dependencies.

In [ ]:
import os, sys
from pathlib import Path

# ── 1. Clone repo (skip if already cloned) ────────────────────────────
REPO_URL  = "https://github.com/Waseem-k/EndoAgents.git"
REPO_NAME = "EndoAgents"

if not Path(REPO_NAME).exists():
    print(f"Cloning {REPO_URL} ...")
    os.system(f"git clone {REPO_URL}")
else:
    print(f"Repo already cloned — pulling latest changes ...")
    os.system(f"cd {REPO_NAME} && git pull")

# ── 2. Set working directory to repo root ─────────────────────────────
REPO_ROOT = Path(REPO_NAME).resolve()
os.chdir(REPO_ROOT)
print(f"Working directory: {os.getcwd()}")

# ── 3. Add repo root to sys.path so all local modules are importable ──
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"sys.path[0]: {sys.path[0]}")

# ── 4. Install dependencies ───────────────────────────────────────────
print("\nInstalling dependencies (this takes ~2 min on first run) ...")
os.system("pip install -q -r requirements.txt")
print("✅ Dependencies installed")

# ── 5. HuggingFace login (Gemma 4 requires licence acceptance) ────────
print("\n🔑 HuggingFace login required for Gemma 4.")
print("   Accept the model licence at: https://huggingface.co/google/gemma-4-E4B-it")
from huggingface_hub import notebook_login
notebook_login()

## Step 1 — Configure

In [ ]:
import torch

# ── Model selection ────────────────────────────────────────────────────
MODEL_ID      = "google/gemma-4-E4B-it"  # change to E2B if VRAM is tight
QUANTISATION  = "4bit"                    # 4bit | 8bit | none
TOKEN_BUDGET  = 560                        # 560 recommended for dense US images
MAX_TOKENS    = 600

# Write .env so config/settings.py picks up overrides
env_txt = (
    f"VISION_MODEL_ID={MODEL_ID}\n"
    f"QUANTISATION={QUANTISATION}\n"
    f"VISUAL_TOKEN_BUDGET={TOKEN_BUDGET}\n"
    f"MAX_NEW_TOKENS={MAX_TOKENS}\n"
)
with open(".env", "w") as f:
    f.write(env_txt)

print(f"Device : {'CUDA ✅' if torch.cuda.is_available() else 'CPU ⚠️  — switch to T4 GPU runtime!'}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")
print(f"Model  : {MODEL_ID}")
print(f"Quant  : {QUANTISATION}")
print(f"Token budget: {TOKEN_BUDGET}")

## Step 2 — Verify imports work

In [ ]:
# This cell catches any remaining import issues before loading the heavy model
try:
    from config.settings import settings
    print(f"✅ config.settings   — model_id={settings.vision_model_id}")
except Exception as e:
    print(f"❌ config.settings   — {e}")

try:
    from models.vision import GemmaVision, CaptionResult
    print(f"✅ models.vision     — GemmaVision, CaptionResult")
except Exception as e:
    print(f"❌ models.vision     — {e}")

try:
    from evaluation.feature_audit import audit_caption, AuditResult
    print(f"✅ evaluation.feature_audit — audit_caption, AuditResult")
except Exception as e:
    print(f"❌ evaluation.feature_audit — {e}")

print("\nAll imports ✅ — proceed to Step 3." if True else "")

## Step 3 — Load Gemma 4
⏳ First load takes 3–5 min (downloading weights). Subsequent runs in the same session are instant.

In [ ]:
from models.vision import GemmaVision

vision = GemmaVision(
    model_id=MODEL_ID,
    quantisation=QUANTISATION,
    visual_token_budget=TOKEN_BUDGET,
)
vision.load()    # downloads + loads model, prints VRAM usage
print(vision)

## Step 4 — Upload ultrasound image

In [ ]:
import io, urllib.request
import matplotlib.pyplot as plt
from PIL import Image

# ── Option A: Load sample image directly from URL ─────────────────────────
SAMPLE_IMAGE_URL = (
    "https://github.com/user-attachments/assets/"
    "3b1c01df-5a42-49de-8fc8-54b92239bffd"
)
USE_SAMPLE = True  # Set to False to upload your own image instead

if USE_SAMPLE:
    print("Downloading sample ultrasound image ...")
    req = urllib.request.Request(SAMPLE_IMAGE_URL,
                                  headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp:
        image = Image.open(io.BytesIO(resp.read())).convert("RGB")
    filename = "sample_ultrasound.jpg"
    print(f"\u2705 Sample image loaded: {image.width}\u00d7{image.height}px")
else:
    # ── Option B: Upload your own image (Colab) ──────────────────────────────
    from google.colab import files
    print("Select your ultrasound image file:")
    uploaded = files.upload()
    filename  = list(uploaded.keys())[0]
    image     = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(image)
ax.axis("off")
ax.set_title(f"{filename}  |  {image.width}\u00d7{image.height}px", fontsize=10)
plt.tight_layout()
plt.show()

# ── Set pathology class for this image ────────────────────────────────────
PATHOLOGY_CLASS = "Adenomyosis"   # Adenomyosis | Fibroid | Normal
print(f"\nPathology class : {PATHOLOGY_CLASS}")
print(f"Image size      : {image.width}\u00d7{image.height}")


## Step 5 — Generate caption

In [ ]:
result = vision.generate_caption(image, max_new_tokens=MAX_TOKENS)

print("=" * 70)
print(f"Model        : {result.model_id}")
print(f"Time         : {result.inference_time_s}s")
print(f"~Tokens out  : {result.approx_tokens}")
print(f"Success      : {result.success}")
if not result.success:
    print(f"Error        : {result.error}")
print("=" * 70)
print("\n📋 GENERATED CAPTION:\n")
print(result.caption)

## Step 6 — Feature coverage audit

In [ ]:
import matplotlib.patches as mpatches
from evaluation.feature_audit import audit_caption

audit = audit_caption(result.caption, pathology_class=PATHOLOGY_CLASS)
print(audit.summary())

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4.5))
names  = list(audit.coverage.keys())
values = [1.0 if v else 0.08 for v in audit.coverage.values()]
colors = ["#2EC4B6" if v else "#ef4444" for v in audit.coverage.values()]
ax.barh(names, values, color=colors, height=0.55, edgecolor="none")
ax.set_xlim(0, 1.3)
ax.set_facecolor("#f8fafc")
fig.patch.set_facecolor("#f8fafc")
ax.set_title(
    f"Gemma 4 Zero-Shot — {audit.score}/{audit.total} features "
    f"({audit.coverage_pct:.0f}%) — {PATHOLOGY_CLASS}",
    fontsize=11, fontweight="bold", pad=10
)
ax.spines[["top", "right", "bottom"]].set_visible(False)
ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
ax.tick_params(axis="y", labelsize=9)
ax.legend(
    handles=[
        mpatches.Patch(color="#2EC4B6", label="Detected"),
        mpatches.Patch(color="#ef4444", label="Missing"),
    ],
    loc="lower right", fontsize=9
)
plt.tight_layout()
plt.show()

if audit.missing:
    print("\n⚠️  Missing features — fine-tuning targets:")
    for m in audit.missing:
        print(f"    — {m}")
else:
    print("\n✅ All required features covered!")

## Step 7 — Compare reference caption (optional)
Paste Dr. Gennari's caption below to get ROUGE-L and BERTScore.

In [ ]:
# Paste the reference caption here — leave empty to skip
REFERENCE_CAPTION = """
"""

if REFERENCE_CAPTION.strip():
    from rouge_score import rouge_scorer
    from bert_score import score as bert_score_fn

    rscorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge   = rscorer.score(REFERENCE_CAPTION.strip(), result.caption)
    P, R, F = bert_score_fn(
        [result.caption], [REFERENCE_CAPTION.strip()],
        lang="en", verbose=False
    )
    print(f"ROUGE-L F1  : {rouge['rougeL'].fmeasure:.4f}")
    print(f"BERTScore F1: {F.item():.4f}")
    print("\n📌 Zero-shot expected range: ROUGE-L 0.10–0.25 | BERTScore 0.70–0.78")
    print("📌 Post fine-tune target    : ROUGE-L > 0.45   | BERTScore > 0.85")
else:
    print("No reference caption provided — skipping metrics.")

## Step 8 — Save & download results

In [ ]:
import json, os
from datetime import datetime
from google.colab import files as colab_files

os.makedirs("evaluation/results", exist_ok=True)

output = {
    "timestamp"            : datetime.now().isoformat(),
    "model_id"             : result.model_id,
    "quantisation"         : result.quantisation,
    "visual_token_budget"  : result.visual_token_budget,
    "image_file"           : filename,
    "pathology_class"      : PATHOLOGY_CLASS,
    "inference_time_s"     : result.inference_time_s,
    "approx_tokens"        : result.approx_tokens,
    "caption"              : result.caption,
    "feature_audit"        : {
        "score"        : audit.score,
        "total"        : audit.total,
        "coverage_pct" : audit.coverage_pct,
        "coverage"     : audit.coverage,
        "missing"      : audit.missing,
    },
}

out_path = "evaluation/results/baseline_gemma4.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"✅ Saved to {out_path}")
colab_files.download(out_path)

print("\n📊 Session Summary")
print(f"  Model          : {result.model_id}")
print(f"  Inference time : {result.inference_time_s}s")
print(f"  Feature score  : {audit.score}/{audit.total} ({audit.coverage_pct:.0f}%)")
print(f"  Missing        : {audit.missing or 'None ✅'}")

---
## 📝 Research Notes

**What this session gives you:**
- The official zero-shot baseline that all fine-tuning improvements will be compared against
- A concrete list of missing features = your fine-tuning targets
- Evidence that general VLMs are insufficient for adenomyosis captioning (research motivation)

**Common issues & fixes:**
| Error | Fix |
|---|---|
| `ModuleNotFoundError: No module named 'models'` | Re-run Step 0 — the `sys.path` fix must run first |
| `ModuleNotFoundError: pydantic_settings` | Re-run Step 0 — `requirements.txt` now includes it |
| CUDA out of memory | Switch to `E2B` model or set `QUANTISATION = "4bit"` |
| Gemma 4 access denied | Accept licence at https://huggingface.co/google/gemma-4-E4B-it |

**Fine-tuning targets:**
```
ROUGE-L   > 0.45   (zero-shot: ~0.10–0.25)
BERTScore > 0.85   (zero-shot: ~0.70–0.78)
Feature coverage = 100% of required sections
```